In [1]:
# ============================================================
# CREDRESOLVE — COUNTERFACTUAL ANALYSIS
# ============================================================
#
# PURPOSE
# ------------------------------------------------------------
# Estimate scenario-based incremental recovery under different
# collection exposure assumptions.
#
# IMPORTANT:
# These are MODEL-BASED COUNTERFACTUAL ESTIMATES.
# They are NOT causal estimates.
#
# AUTHORITATIVE RECOVERY DEFINITION
# ------------------------------------------------------------
# recovered_account:
#     Account has at least one SUCCESS payment.
#
# recovery_amount:
#     SUM(amount) from SUCCESS payments only,
#     after payment_id deduplication.
#
# FAILED / PENDING / REVERSED payments are NOT recovery.
# ============================================================


from pathlib import Path
import pandas as pd
import numpy as np


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_ROOT = Path.cwd().parent

GOLDEN_DIR = (
    PROJECT_ROOT
    / "data"
    / "golden"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 90)
print("CREDRESOLVE — COUNTERFACTUAL ANALYSIS")
print("=" * 90)


# ============================================================
# 2. LOAD GOLDEN DATA
# ============================================================

golden_files = sorted(
    GOLDEN_DIR.glob("*_golden.csv")
)

golden = {
    file.stem.replace("_golden", ""):
    pd.read_csv(file)
    for file in golden_files
}


if "accounts" not in golden:

    raise ValueError(
        "accounts_golden.csv not found"
    )


if "payments" not in golden:

    raise ValueError(
        "payments_golden.csv not found"
    )


accounts = golden["accounts"].copy()

payments = golden["payments"].copy()


print(
    "Accounts loaded:",
    f"{len(accounts):,}"
)


# ============================================================
# 3. VALIDATE REQUIRED COLUMNS
# ============================================================

required_account_columns = [

    "account_id"

]


required_payment_columns = [

    "payment_id",
    "account_id",
    "payment_status",
    "amount"

]


missing_accounts = [

    column

    for column in required_account_columns

    if column not in accounts.columns

]


missing_payments = [

    column

    for column in required_payment_columns

    if column not in payments.columns

]


if missing_accounts:

    raise ValueError(
        "Missing account columns: "
        f"{missing_accounts}"
    )


if missing_payments:

    raise ValueError(
        "Missing payment columns: "
        f"{missing_payments}"
    )


print(
    "Required columns validated."
)


# ============================================================
# 4. CLEAN PAYMENT FIELDS
# ============================================================

payments["payment_status"] = (

    payments[
        "payment_status"
    ]

    .astype("string")

    .str.upper()

    .str.strip()

)


payments["amount"] = pd.to_numeric(

    payments["amount"],

    errors="coerce"

).fillna(0)


# ============================================================
# 5. DEDUPLICATE PAYMENT IDs
# ============================================================
#
# IMPORTANT:
# The same payment_id must only contribute once.
#
# When event_at exists, retain the earliest record.
# ============================================================

print()
print("=" * 90)
print("PAYMENT DEDUPLICATION")
print("=" * 90)


payment_rows_before = len(
    payments
)


if "event_at" in payments.columns:

    payments["event_at"] = pd.to_datetime(

        payments["event_at"],

        errors="coerce"

    )

    payments = (

        payments

        .sort_values(

            [
                "payment_id",
                "event_at"
            ],

            na_position="last"

        )

        .drop_duplicates(

            subset=[
                "payment_id"
            ],

            keep="first"

        )

    )

else:

    payments = (

        payments

        .drop_duplicates(

            subset=[
                "payment_id"
            ],

            keep="first"

        )

    )


payment_rows_after = len(
    payments
)


print(
    "Payment rows before deduplication:",
    f"{payment_rows_before:,}"
)


print(
    "Payment rows after deduplication:",
    f"{payment_rows_after:,}"
)


print(
    "Duplicate payment rows removed:",
    f"{payment_rows_before - payment_rows_after:,}"
)


# ============================================================
# 6. SUCCESS-ONLY RECOVERY
# ============================================================

payments["is_success"] = (

    payments[
        "payment_status"
    ]

    == "SUCCESS"

)


payments["success_recovery_amount"] = np.where(

    payments["is_success"],

    payments["amount"],

    0.0

)


# ============================================================
# 7. ACCOUNT-LEVEL RECOVERY
# ============================================================

payment_account = (

    payments

    .groupby(
        "account_id"
    )

    .agg(

        successful_payment_count=(

            "is_success",

            "sum"

        ),

        recovery_amount=(

            "success_recovery_amount",

            "sum"

        )

    )

    .reset_index()

)


payment_account["recovered_account"] = (

    payment_account[
        "successful_payment_count"
    ]

    > 0

).astype(int)


# ============================================================
# 8. BUILD COUNTERFACTUAL ACCOUNT DATASET
# ============================================================

analysis = accounts.merge(

    payment_account,

    on="account_id",

    how="left"

)


analysis[
    "successful_payment_count"
] = (

    analysis[
        "successful_payment_count"
    ]

    .fillna(0)

)


analysis[
    "recovery_amount"
] = (

    analysis[
        "recovery_amount"
    ]

    .fillna(0)

)


analysis[
    "recovered_account"
] = (

    analysis[
        "recovered_account"
    ]

    .fillna(0)

    .astype(int)

)


# ============================================================
# 9. BASELINE PORTFOLIO
# ============================================================

accounts_count = int(

    analysis[
        "account_id"
    ]

    .nunique()

)


recovered_accounts = int(

    analysis[
        "recovered_account"
    ]

    .sum()

)


unrecovered_accounts = (

    accounts_count
    -
    recovered_accounts

)


baseline_recovery_rate = (

    recovered_accounts
    /
    accounts_count

    if accounts_count > 0

    else 0

)


baseline_recovery_amount = float(

    analysis[
        "recovery_amount"
    ]

    .sum()

)


if "outstanding_amount" in analysis.columns:

    baseline_outstanding_amount = float(

        pd.to_numeric(

            analysis[
                "outstanding_amount"
            ],

            errors="coerce"

        )

        .fillna(0)

        .sum()

    )

else:

    baseline_outstanding_amount = np.nan


average_recovery_per_recovered_account = (

    baseline_recovery_amount
    /
    recovered_accounts

    if recovered_accounts > 0

    else 0

)


baseline = pd.DataFrame([{

    "accounts":
        accounts_count,

    "recovered_accounts":
        recovered_accounts,

    "unrecovered_accounts":
        unrecovered_accounts,

    "baseline_recovery_rate":
        baseline_recovery_rate,

    "baseline_recovery_amount":
        baseline_recovery_amount,

    "baseline_outstanding_amount":
        baseline_outstanding_amount,

    "average_recovery_per_recovered_account":
        average_recovery_per_recovered_account

}])


print()
print("=" * 90)
print("BASELINE PORTFOLIO")
print("=" * 90)

print(
    baseline.to_string(
        index=False
    )
)


# ============================================================
# 10. BASELINE VALIDATION
# ============================================================

print()
print("=" * 90)
print("BASELINE VALIDATION")
print("=" * 90)


print(
    "Expected recovered accounts: 13,284"
)


print(
    "Observed recovered accounts:",
    recovered_accounts
)


print(
    "Expected recovery rate: 44.28%"
)


print(
    "Observed recovery rate:",
    round(
        baseline_recovery_rate * 100,
        2
    ),
    "%"
)


print(
    "Authoritative SUCCESS recovery amount:",
    round(
        baseline_recovery_amount,
        2
    )
)


# ============================================================
# 11. CREATE EXPOSURE FEATURES
# ============================================================

# The counterfactual model needs exposure variables.
#
# If these fields exist in accounts, use them.
# Otherwise derive them from the available Golden datasets.


# ------------------------------------------------------------
# CALL EXPOSURE
# ------------------------------------------------------------

if "calls" in golden:

    calls = golden["calls"].copy()

    if "account_id" in calls.columns:

        call_id_column = (

            "call_id"

            if "call_id" in calls.columns

            else None

        )

        if call_id_column:

            call_exposure = (

                calls

                .groupby(
                    "account_id"
                )

                .agg(

                    total_calls=(

                        call_id_column,

                        "nunique"

                    )

                )

                .reset_index()

            )

        else:

            call_exposure = (

                calls

                .groupby(
                    "account_id"
                )

                .size()

                .reset_index(
                    name="total_calls"
                )

            )

    else:

        call_exposure = pd.DataFrame(

            columns=[
                "account_id",
                "total_calls"
            ]

        )

else:

    call_exposure = pd.DataFrame(

        columns=[
            "account_id",
            "total_calls"
        ]

    )


analysis = analysis.merge(

    call_exposure,

    on="account_id",

    how="left"

)


analysis["total_calls"] = (

    analysis[
        "total_calls"
    ]

    .fillna(0)

)


# ------------------------------------------------------------
# ATTEMPT EXPOSURE
# ------------------------------------------------------------

if "call_attempts" in golden:

    attempts = golden[
        "call_attempts"
    ].copy()

    if "account_id" in attempts.columns:

        attempt_id_column = (

            "attempt_id"

            if "attempt_id" in attempts.columns

            else None

        )

        if attempt_id_column:

            attempt_exposure = (

                attempts

                .groupby(
                    "account_id"
                )

                .agg(

                    total_attempts=(

                        attempt_id_column,

                        "nunique"

                    )

                )

                .reset_index()

            )

        else:

            attempt_exposure = (

                attempts

                .groupby(
                    "account_id"
                )

                .size()

                .reset_index(
                    name="total_attempts"
                )

            )

    else:

        attempt_exposure = pd.DataFrame(

            columns=[
                "account_id",
                "total_attempts"
            ]

        )

else:

    attempt_exposure = pd.DataFrame(

        columns=[
            "account_id",
            "total_attempts"
        ]

    )


analysis = analysis.merge(

    attempt_exposure,

    on="account_id",

    how="left"

)


analysis["total_attempts"] = (

    analysis[
        "total_attempts"
    ]

    .fillna(0)

)


# ============================================================
# 12. EXPOSURE GROUPS
# ============================================================

analysis["call_exposure_group"] = pd.cut(

    analysis[
        "total_calls"
    ],

    bins=[
        -np.inf,
        2,
        4,
        np.inf
    ],

    labels=[
        "Low",
        "Medium",
        "High"
    ]

)


analysis["attempt_exposure_group"] = pd.cut(

    analysis[
        "total_attempts"
    ],

    bins=[
        -np.inf,
        2,
        5,
        np.inf
    ],

    labels=[
        "Low",
        "Medium",
        "High"
    ]

)


# ============================================================
# 13. OBSERVED EXPOSURE RECOVERY
# ============================================================

def exposure_summary(

    dataframe,
    group_column

):

    result = (

        dataframe

        .groupby(
            group_column,
            observed=False
        )

        .agg(

            accounts=(

                "account_id",
                "nunique"

            ),

            recovered_accounts=(

                "recovered_account",
                "sum"

            ),

            recovery_amount=(

                "recovery_amount",
                "sum"

            )

        )

        .reset_index()

    )


    result[
        "recovery_rate"
    ] = (

        result[
            "recovered_accounts"
        ]

        /

        result[
            "accounts"
        ]

    )


    return result


call_summary = exposure_summary(

    analysis,

    "call_exposure_group"

)


attempt_summary = exposure_summary(

    analysis,

    "attempt_exposure_group"

)


print()
print("=" * 90)
print("CALL EXPOSURE")
print("=" * 90)

print(
    call_summary.to_string(
        index=False
    )
)


print()
print("=" * 90)
print("ATTEMPT EXPOSURE")
print("=" * 90)

print(
    attempt_summary.to_string(
        index=False
    )
)


# ============================================================
# 14. ESTIMATE SCENARIO EFFECTS
# ============================================================
#
# IMPORTANT:
# These are NOT causal effects.
#
# The model estimates a scenario by comparing exposure groups
# with the observed baseline.
#
# We use conservative scenario effects rather than claiming
# that changing exposure will definitely cause recovery.
# ============================================================


baseline_rate = baseline_recovery_rate


def group_rate(

    summary,
    group_name

):

    row = summary[
        summary[
            summary.columns[0]
        ].astype(str)
        == group_name
    ]


    if len(row) == 0:

        return np.nan


    return float(
        row.iloc[0][
            "recovery_rate"
        ]
    )


low_call_rate = group_rate(

    call_summary,

    "Low"

)


medium_call_rate = group_rate(

    call_summary,

    "Medium"

)


high_call_rate = group_rate(

    call_summary,

    "High"

)


low_attempt_rate = group_rate(

    attempt_summary,

    "Low"

)


medium_attempt_rate = group_rate(

    attempt_summary,

    "Medium"

)


high_attempt_rate = group_rate(

    attempt_summary,

    "High"

)


# ============================================================
# 15. SCENARIO DEFINITIONS
# ============================================================

# Telephony:
# Compare high call exposure with low call exposure.
#
# Agent:
# Use attempt exposure as a descriptive operational proxy.
#
# Targeting:
# No causal targeting effect is assumed unless targeting data
# can be directly linked to account recovery.
#
# Therefore targeting remains conservative.
# ============================================================


telephony_effect = (

    high_call_rate
    -
    low_call_rate

)


agent_effect = (

    high_attempt_rate
    -
    low_attempt_rate

)


if pd.isna(telephony_effect):

    telephony_effect = 0


if pd.isna(agent_effect):

    agent_effect = 0


targeting_effect = 0.0


# ------------------------------------------------------------
# SCENARIO UPLIFT
# ------------------------------------------------------------

scenario_effects = pd.DataFrame([

    {

        "scenario":
            "A",

        "description":
            "Conservative",

        "driver":
            "Telephony",

        "model_effect_pp":
            telephony_effect * 100

    },

    {

        "scenario":
            "B",

        "description":
            "Targeted Opportunity",

        "driver":
            "Collection Agents",

        "model_effect_pp":
            agent_effect * 100

    },

    {

        "scenario":
            "C",

        "description":
            "Upside",

        "driver":
            "Borrower Targeting",

        "model_effect_pp":
            targeting_effect * 100

    }

])


# ============================================================
# 16. SCENARIO ECONOMICS
# ============================================================

scenario_rows = []


for _, row in scenario_effects.iterrows():

    effect_pp = float(
        row["model_effect_pp"]
    )


    incremental_rate = (

        effect_pp / 100

    )


    estimated_incremental_recovered_accounts = (

        accounts_count
        *
        incremental_rate

    )


    estimated_incremental_recovery_amount = (

        baseline_outstanding_amount
        *
        incremental_rate

        if pd.notna(
            baseline_outstanding_amount
        )

        else 0

    )


    scenario_rows.append({

        "scenario":
            row["scenario"],

        "description":
            row["description"],

        "driver":
            row["driver"],

        "eligible_accounts":
            accounts_count,

        "baseline_recovery_rate":
            baseline_rate,

        "estimated_recovery_rate":
            baseline_rate
            +
            incremental_rate,

        "model_effect_pp":
            effect_pp,

        "estimated_incremental_recovered_accounts":
            estimated_incremental_recovered_accounts,

        "estimated_incremental_recovery_amount":
            estimated_incremental_recovery_amount,

        "causal_claim":
            False

    })


scenarios = pd.DataFrame(
    scenario_rows
)


# ============================================================
# 17. SCENARIO SUMMARY
# ============================================================

print()
print("=" * 90)
print("COUNTERFACTUAL SCENARIOS")
print("=" * 90)

print(

    scenarios.to_string(
        index=False
    )

)


# ============================================================
# 18. LOW-EXPOSURE COUNTERFACTUAL
# ============================================================

low_exposure = pd.DataFrame([{

    "baseline_recovery_rate":
        baseline_rate,

    "low_call_recovery_rate":
        low_call_rate,

    "low_attempt_recovery_rate":
        low_attempt_rate,

    "telephony_effect_pp":
        telephony_effect * 100,

    "agent_effect_pp":
        agent_effect * 100,

    "causal_interpretation":
        "NOT ESTABLISHED"

}])


# ============================================================
# 19. SENSITIVITY ANALYSIS
# ============================================================

sensitivity_rows = []


uplift_multipliers = [

    0.50,
    0.75,
    1.00,
    1.25,
    1.50

]


for _, row in scenarios.iterrows():

    base_value = float(

        row[
            "estimated_incremental_recovery_amount"
        ]

    )


    for multiplier in uplift_multipliers:

        sensitivity_rows.append({

            "scenario":
                row["scenario"],

            "description":
                row["description"],

            "uplift_multiplier":
                multiplier,

            "base_incremental_recovery_amount":
                base_value,

            "sensitivity_incremental_recovery_amount":
                base_value
                *
                multiplier,

            "causal_claim":
                False

        })


sensitivity = pd.DataFrame(
    sensitivity_rows
)


# ============================================================
# 20. ACCOUNT-LEVEL COUNTERFACTUAL DATASET
# ============================================================

account_level = analysis[

    [

        "account_id",

        "recovered_account",

        "recovery_amount",

        "total_calls",

        "total_attempts",

        "call_exposure_group",

        "attempt_exposure_group"

    ]

].copy()


# ============================================================
# 21. ANALYSIS SUMMARY
# ============================================================

analysis_summary = pd.DataFrame([{

    "accounts":
        accounts_count,

    "recovered_accounts":
        recovered_accounts,

    "baseline_recovery_rate":
        baseline_rate,

    "baseline_recovery_amount":
        baseline_recovery_amount,

    "baseline_outstanding_amount":
        baseline_outstanding_amount,

    "telephony_model_effect_pp":
        telephony_effect * 100,

    "agent_model_effect_pp":
        agent_effect * 100,

    "targeting_model_effect_pp":
        targeting_effect * 100,

    "causal_claim":
        False,

    "raw_source_modified":
        False

}])


# ============================================================
# 22. ASSUMPTIONS
# ============================================================

assumptions = pd.DataFrame([

    {

        "assumption":
            "Recovery definition",

        "value":
            "SUCCESS payments only",

        "interpretation":
            "Authoritative recovery definition"

    },

    {

        "assumption":
            "Payment deduplication",

        "value":
            "One record per payment_id",

        "interpretation":
            "Prevents duplicate payment inflation"

    },

    {

        "assumption":
            "Scenario effects",

        "value":
            "Observed exposure associations",

        "interpretation":
            "Not causal estimates"

    },

    {

        "assumption":
            "Targeting effect",

        "value":
            "0 pp",

        "interpretation":
            "No defensible causal targeting effect established"

    },

    {

        "assumption":
            "Investment decision",

        "value":
            "Requires controlled validation",

        "interpretation":
            "Counterfactual outputs should not be treated as guaranteed recovery"

    }

])


# ============================================================
# 23. SAVE OUTPUTS
# ============================================================

baseline.to_csv(

    OUTPUT_DIR
    /
    "counterfactual_baseline.csv",

    index=False

)


scenarios.to_csv(

    OUTPUT_DIR
    /
    "counterfactual_scenarios.csv",

    index=False

)


sensitivity.to_csv(

    OUTPUT_DIR
    /
    "counterfactual_sensitivity.csv",

    index=False

)


low_exposure.to_csv(

    OUTPUT_DIR
    /
    "counterfactual_low_exposure.csv",

    index=False

)


analysis_summary.to_csv(

    OUTPUT_DIR
    /
    "counterfactual_analysis_summary.csv",

    index=False

)


assumptions.to_csv(

    OUTPUT_DIR
    /
    "counterfactual_assumptions.csv",

    index=False

)


account_level.to_csv(

    OUTPUT_DIR
    /
    "counterfactual_account_level_dataset.csv",

    index=False

)


# ============================================================
# 24. FINAL VALIDATION
# ============================================================

print()
print("=" * 90)
print("FINAL COUNTERFACTUAL VALIDATION")
print("=" * 90)


print(
    "Accounts:",
    f"{accounts_count:,}"
)


print(
    "Recovered accounts:",
    f"{recovered_accounts:,}"
)


print(
    "Recovery rate:",
    round(
        baseline_rate * 100,
        2
    ),
    "%"
)


print(
    "SUCCESS-only recovery amount:",
    round(
        baseline_recovery_amount,
        2
    )
)


print(
    "Expected authoritative recovery amount:",
    "approximately ₹1.315584 billion"
)


# ============================================================
# 25. FINAL STATUS
# ============================================================

print()
print("=" * 90)
print("COUNTERFACTUAL ANALYSIS COMPLETE")
print("=" * 90)

print(
    "Recovery definition: SUCCESS payments only"
)

print(
    "Payment IDs deduplicated: TRUE"
)

print(
    "Causal claims made: FALSE"
)

print(
    "Raw source data modified: FALSE"
)

print(
    "Outputs saved successfully."
)

CREDRESOLVE — COUNTERFACTUAL ANALYSIS
Accounts loaded: 30,000
Required columns validated.

PAYMENT DEDUPLICATION
Payment rows before deduplication: 25,500
Payment rows after deduplication: 25,000
Duplicate payment rows removed: 500

BASELINE PORTFOLIO
 accounts  recovered_accounts  unrecovered_accounts  baseline_recovery_rate  baseline_recovery_amount  baseline_outstanding_amount  average_recovery_per_recovered_account
    30000               13284                 16716                  0.4428              1.315584e+09                 1.048904e+10                             99035.22769

BASELINE VALIDATION
Expected recovered accounts: 13,284
Observed recovered accounts: 13284
Expected recovery rate: 44.28%
Observed recovery rate: 44.28 %
Authoritative SUCCESS recovery amount: 1315583964.64

CALL EXPOSURE
call_exposure_group  accounts  recovered_accounts  recovery_amount  recovery_rate
                Low     12637                5547     549853035.54       0.438949
             Medium